# CCCOH: GENIE covariance matrices

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
syst_name = ("GENIE",)

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

import uproot

import sys
sys.path.append('../../../')
from pyanalib.split_df_helpers import *
import pyanalib.pandas_helpers as ph
from analysis_village.unfolding.covariance import *
from analysis_village.unfolding.utils import *

from cohpi_topologies import *
from var_config_cohpi import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

## 1. Open MC df

### 1 - 1: File path and name

In [ ]:
input_path = "/data/sungbino/sbnd/v10_14_02/cohpi/"

## -- this file is based on 
####    geniewgtdf = geniesyst.geniesyst(f, nudf.ind, multisim_nuniv=100, slim=True, systematics=None)
####    bnbwgtdf = bnbsyst.bnbsyst(f, nudf.ind, multisim_nuniv=100, slim=True)
mc_file_name = "mc_MCP2025C_FallProduction_prodgenie_corsika_proton_rockbox0p1_sbnd_CV_v10_14_02_flatcaf_sbnd_cohpi_wienersvd.df"

mc_file_path = path.join(input_path, mc_file_name)

### 1 - 2: Check df keys and n_split

In [ ]:
print("n split", get_n_split(mc_file_path))
print("Keys in mc file:")
print_keys(mc_file_path)

### 1 - 3: Load the dfs

Note:
- mc_evt_df is skimmed df with a single level for columns.
- mc_mcnu_df is skimmed df but with two levels for columns. second level is for multiverse weights

In [ ]:
mc_keys2load = ['hdr', "mcnu", 'evt']
mc_dfs = load_dfs(mc_file_path, mc_keys2load, n_max_concat=10)
mc_evt_df = mc_dfs['evt']
mc_hdr_df = mc_dfs['hdr']
mc_mcnu_df = mc_dfs['mcnu']

In [ ]:
mc_evt_df.columns

In [ ]:
mc_mcnu_df.columns[0:10]

## 2. Normalization Variables

POT

In [ ]:
mc_tot_pot = mc_hdr_df['pot'].sum()
print("Total MC POT:", mc_tot_pot)
mc_pot_scale = 1.
mc_evt_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_evt_df))

tot_pot = mc_tot_pot

Volume

In [ ]:
TPC_Vol = (180. * 2.) * (190. * 2) * (250. - 10.) + (180. * 2.) * (190. + 100.) * (450. - 250.) # cm^3

Flux

In [ ]:
fluxfile = "/data/sungbino/sbnd/flux/sbnd_original_flux.root"
flux = uproot.open(fluxfile)
numu_flux = flux["flux_sbnd_numu"].to_numpy()
anumu_flux = flux["flux_sbnd_anumu"].to_numpy()
numu_flux_vals = numu_flux[0]
anumu_flux_vals = anumu_flux[0]

INTEGRATED_FLUX = (numu_flux_vals.sum() + anumu_flux_vals.sum()) / (1e4  * 1e6) # to cm2 # to POT
print("Integrated flux (numu + anumu) over total MC POT: %.3e per (cm^2 POT)" % INTEGRATED_FLUX)

## 3. match mc_evt_df and mc_mcnu_df

In [ ]:
mc_evt_df = ph.multicol_merge(mc_evt_df.reset_index(), mc_mcnu_df.reset_index(),
                               left_on=[("__ntuple", ""), ("entry", ""), ("tmatch_idx", "")],
                               right_on=[("__ntuple", ""), ("entry", ""), ("rec.mc.nu..index", "")],
                               how="left") ## -- save all sllices
mc_evt_df.loc[mc_evt_df['nuint_categ'].isna(), ['nuint_categ']] = -2

In [ ]:
mc_evt_df.nuint_categ.value_counts()

## 4. Spiltting to signal region (SR) and sideband (SB)

- Signal region: reco |t| < 0.04 (GeV/c)^2
- Sideband: 0.06 (GeV/c)^2 < reco |t| < 0.16 (GeV/c)^2

In [ ]:
mc_evt_df_sr = mc_evt_df[mc_evt_df.reco_t < 0.04]
mc_evt_df_sb = mc_evt_df[(mc_evt_df.reco_t > 0.06) & (mc_evt_df.reco_t < 0.16)]

## 5. Cross Section Variable Configs

P_mu

In [ ]:
mc_evt_df

## 5. Signal histograms

"All Signal in MC" are the same in SR and SB

In [ ]:
signal_hists(evtdf=mc_evt_df_sr, nudf=mc_mcnu_df, var_config=varcfg_p_mu, save_fig=False, save_name=None)
signal_hists(evtdf=mc_evt_df_sb, nudf=mc_mcnu_df, var_config=varcfg_p_mu, save_fig=False, save_name=None)

## 6. Collect Histograms in CV and Multiverses

### 6 - 1: collect events in CV + multiverses, in each region

In [ ]:
cov_type = "xsec"
sr_signal_univ_events, sr_signal_sel_reco_cv, sr_bkg_univ_events, sr_bkg_sec_rec_cv = get_genie_univs(cov_type, mc_evt_df_sr, mc_mcnu_df, varcfg_p_mu, syst_name, flux=INTEGRATED_FLUX, pot=tot_pot, vol=TPC_Vol, topology_list=mode_list)
sb_signal_univ_events, sb_signal_sel_reco_cv, sb_bkg_univ_events, sb_bkg_sec_rec_cv = get_genie_univs(cov_type, mc_evt_df_sb, mc_mcnu_df, varcfg_p_mu, syst_name, flux=INTEGRATED_FLUX, pot=tot_pot, vol=TPC_Vol, topology_list=mode_list)

### 6 - 2: plot CV vs. Multiverse plots in SR

#### 6 - 2 - a: Signal

In [ ]:
plot_univ_hists(sr_signal_univ_events, sr_signal_sel_reco_cv, "GENIE", varcfg_p_mu, categ_name="Signal")

#### 6 - 2 - b: backgrounds

In [ ]:
for i in range(len(mode_list))[1:]:
    this_mode_name = mode_labels[i]
    this_categ_univ_events = sr_bkg_univ_events[:, i - 1, :]
    plot_univ_hists(this_categ_univ_events, sr_bkg_sec_rec_cv[i - 1], "GENIE", varcfg_p_mu, categ_name=this_mode_name)

### 6 - 2: plot CV vs. Multiverse plots in SB

#### 6 - 2 - a: Signal

In [ ]:
plot_univ_hists(sb_signal_univ_events, sb_signal_sel_reco_cv, "GENIE", varcfg_p_mu, categ_name="Signal")

#### 6 - 2 - b: Background

In [ ]:
for i in range(len(mode_list))[1:]:
    this_mode_name = mode_labels[i]
    this_categ_univ_events = sr_bkg_univ_events[:, i - 1, :]
    plot_univ_hists(this_categ_univ_events, sr_bkg_sec_rec_cv[i - 1], "GENIE", varcfg_p_mu, categ_name=this_mode_name)

## 7. Covariance Matrices

For CCBC, we consider signal events (m) and background events (b) in signal region (s) and sideband (c)

We need these self-covariance matrices
1. Cov(m_s, m_s): cov_ms_ms
2. Cov(b_s, b_s): cov_bs_bs
3. Cov(m_c, m_c): cov_mc_mc
4. Cov(b_c, b_c): cov_bc_bc

In addition, we also need

5. Cov(m_s, b_s), Cov(b_s, m_s): cov_ms_bs, cov_bs_ms
6. Cov(m_s, m_c), Cov(m_c, m_s): cov_ms_mc, cov_mc_ms
7. Cov(m_s, b_c), Cov(b_c, m_s): cov_ms_bc, cov_bc_ms
8. Cov(b_s, m_c), Cov(m_c, b_s): cov_bs_mc, cov_mc_bs
9. Cov(b_s, b_c), Cov(b_c, b_s): cov_bs_bc, cov_bc_bs
10. Cov(m_c, b_c), Cov(b_c, m_c): cov_mc_bc, cov_bc_mc

### 7 - 1: Signal region, cov_ms_ms and cov_bs_bs

In [ ]:
cov_ms_ms = get_covariance_matrix_self(sr_signal_univ_events, sr_signal_sel_reco_cv)
plot_heatmap(cov_ms_ms["cov"], varcfg_p_mu, title="GENIE syst. Cov. Signal in SR")

In [ ]:
### For the total background
sr_total_bkg_sec_rec_cv = sr_bkg_sec_rec_cv.sum(axis=0)
sr_total_bkg_univ_events = sr_bkg_univ_events.sum(axis=1)
cov_bs_bs = get_covariance_matrix_self(sr_total_bkg_univ_events, sr_total_bkg_sec_rec_cv)
plot_heatmap(cov_bs_bs["cov"], varcfg_p_mu, title="GENIE syst. Cov. Background in SR")

### 7 - 2: Sideband, cov_mc_mc and cov_bc_bc

In [ ]:
cov_mc_mc = get_covariance_matrix_self(sb_signal_univ_events, sb_signal_sel_reco_cv)
plot_heatmap(cov_mc_mc["cov"], varcfg_p_mu, title="GENIE syst. Cov. Signal in SB")

In [ ]:
### For the total background
sb_total_bkg_sec_rec_cv = sb_bkg_sec_rec_cv.sum(axis=0)
sb_total_bkg_univ_events = sb_bkg_univ_events.sum(axis=1)
cov_bc_bc = get_covariance_matrix_self(sb_total_bkg_univ_events, sb_total_bkg_sec_rec_cv)
plot_heatmap(cov_bc_bc["cov"], varcfg_p_mu, title="GENIE syst. Cov. Background in SB")

### 7 - 3: Crossing covariance matrices

In [ ]:
cov_ms_bs = get_covariance_matrix(sr_signal_univ_events, sr_signal_sel_reco_cv, sr_total_bkg_univ_events, sr_total_bkg_sec_rec_cv)
cov_bs_ms = get_covariance_matrix(sr_total_bkg_univ_events, sr_total_bkg_sec_rec_cv, sr_signal_univ_events, sr_signal_sel_reco_cv)

cov_ms_mc = get_covariance_matrix(sr_signal_univ_events, sr_signal_sel_reco_cv, sb_signal_univ_events, sb_signal_sel_reco_cv)
cov_mc_ms = get_covariance_matrix(sb_signal_univ_events, sb_signal_sel_reco_cv, sr_signal_univ_events, sr_signal_sel_reco_cv)

cov_ms_bc = get_covariance_matrix(sr_signal_univ_events, sr_signal_sel_reco_cv, sb_total_bkg_univ_events, sb_total_bkg_sec_rec_cv)
cov_bc_ms = get_covariance_matrix(sb_total_bkg_univ_events, sb_total_bkg_sec_rec_cv, sr_signal_univ_events, sr_signal_sel_reco_cv)

cov_bs_mc = get_covariance_matrix(sr_total_bkg_univ_events, sr_total_bkg_sec_rec_cv, sb_signal_univ_events, sb_signal_sel_reco_cv)
cov_mc_bs = get_covariance_matrix(sb_signal_univ_events, sb_signal_sel_reco_cv, sr_total_bkg_univ_events, sr_total_bkg_sec_rec_cv)

cov_bs_bc = get_covariance_matrix(sr_total_bkg_univ_events, sr_total_bkg_sec_rec_cv, sb_total_bkg_univ_events, sb_total_bkg_sec_rec_cv)
cov_bc_bs = get_covariance_matrix(sb_total_bkg_univ_events, sb_total_bkg_sec_rec_cv, sr_total_bkg_univ_events, sr_total_bkg_sec_rec_cv)

cov_mc_bc = get_covariance_matrix(sb_signal_univ_events, sb_signal_sel_reco_cv, sb_total_bkg_univ_events, sb_total_bkg_sec_rec_cv)
cov_bc_mc = get_covariance_matrix(sb_total_bkg_univ_events, sb_total_bkg_sec_rec_cv, sb_signal_univ_events, sb_signal_sel_reco_cv)

## 8. Save CV hists and Cov matrices

In [ ]:
ret_dict = {
    "ms": sr_signal_sel_reco_cv,
    "bs": sr_total_bkg_sec_rec_cv,
    "mc": sb_signal_sel_reco_cv,
    "bc": sb_total_bkg_sec_rec_cv,
    "cov_ms_ms": cov_ms_ms,
    "cov_bs_bs": cov_bs_bs,
    "cov_mc_mc": cov_mc_mc,
    "cov_bc_bc": cov_bc_bc,
    "cov_ms_bs": cov_ms_bs,
    "cov_bs_ms": cov_bs_ms,
    "cov_ms_mc": cov_ms_mc,
    "cov_mc_ms": cov_mc_ms,
    "cov_ms_bc": cov_ms_bc,
    "cov_bc_ms": cov_bc_ms,
    "cov_bs_mc": cov_bs_mc,
    "cov_mc_bs": cov_mc_bs,
    "cov_bs_bc": cov_bs_bc,
    "cov_bc_bs": cov_bc_bs,
    "cov_mc_bc": cov_mc_bc,
    "cov_bc_mc": cov_bc_mc,
}

np.savez("genie_syst_cov_matrices.npz", **ret_dict)